# Explainability and aggregate figures

**Scientific purpose.** Generate privacy-safe aggregate result figures and document the
historical W4 Grad-CAM/phase-drop interfaces with correct interpretation boundaries.

**Runnability classification:** the internal-performance figure and external aggregate-table
inspection are runnable from a clean clone. The complete released aggregate figure set is
generated by `scripts/generate_aggregate_figures.py`.
Patient-level explainability requires private processed volumes and retained W4
checkpoints; a complete exact five-fold recreation is unavailable because historical W4
fold 0 was not retained.

**Inputs:** released aggregate result tables; private W4 arrays/checkpoints for case-level
inspection. **Notebook output:** `results/figures/internal_performance.png`; the external
table is displayed without writing an additional figure.

Grad-CAM localizes evidence within each independently lesion-centered input channel.
Phase-drop measures prediction sensitivity to a channel; neither establishes registered
voxel correspondence or direct temporal tracking across phases.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as torch_functional
from monai.networks.nets import resnet18

FIGURE_ROOT = REPO_ROOT / "results" / "figures"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)


## Historical W4 explainability interface

The model definition matches W4. The utility accepts an already-authorized model and
tensor; it does not search for or fabricate a missing checkpoint.


In [ ]:
def make_w4_model():
    return resnet18(
        spatial_dims=3,
        n_input_channels=4,
        num_classes=3,
        feed_forward=True,
    )


class GradCAM3D:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(self._capture_activations)
        self.backward_handle = target_layer.register_full_backward_hook(self._capture_gradients)

    def _capture_activations(self, _module, _inputs, output):
        self.activations = output.detach()

    def _capture_gradients(self, _module, _gradient_input, gradient_output):
        self.gradients = gradient_output[0].detach()

    def __call__(self, image, class_index: int):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(image)
        logits[:, class_index].sum().backward()
        if self.activations is None or self.gradients is None:
            raise RuntimeError("Grad-CAM hooks did not capture activations and gradients.")
        weights = self.gradients.mean(dim=(2, 3, 4), keepdim=True)
        heatmap = torch_functional.relu((weights * self.activations).sum(dim=1, keepdim=True))
        heatmap = torch_functional.interpolate(
            heatmap, size=image.shape[2:], mode="trilinear", align_corners=False
        )
        maximum = heatmap.amax(dim=(2, 3, 4), keepdim=True).clamp_min(1e-8)
        return (heatmap / maximum).detach().cpu().numpy()

    def close(self):
        self.forward_handle.remove()
        self.backward_handle.remove()


@torch.no_grad()
def phase_drop_probabilities(model, image):
    baseline = torch.softmax(model(image), dim=1)
    dropped = []
    for channel in range(image.shape[1]):
        ablated = image.clone()
        ablated[:, channel] = 0
        dropped.append(torch.softmax(model(ablated), dim=1))
    return baseline.cpu().numpy(), torch.stack(dropped).cpu().numpy()


## Privacy-safe internal performance figure

The plot uses only the released aggregate table. It does not display patient images,
masks, identifiers, confusion matrices derived from undistributed rows, or qualitative
case panels.


In [ ]:
performance_path = REPO_ROOT / "results" / "aggregate" / "internal_performance.csv"
if not performance_path.is_file():
    raise FileNotFoundError("Released aggregate internal performance table is unavailable.")
performance = pd.read_csv(performance_path)

model_column = "model" if "model" in performance.columns else "Model"
test_auc_column = next(
    column for column in performance.columns
    if column.lower().replace(" ", "_") in {"test_auc", "held_out_auc", "heldout_macro_auc", "internal_evaluation_auc"}
)
figure, axis = plt.subplots(figsize=(9, 4.8))
axis.bar(performance[model_column], performance[test_auc_column], color="#315f8c")
axis.set_ylabel("Held-out macro one-vs-rest AUC")
axis.set_ylim(0.5, 1.0)
axis.tick_params(axis="x", rotation=30)
axis.set_title("Internal evaluation performance")
figure.tight_layout()
figure.savefig(FIGURE_ROOT / "internal_performance.png", dpi=200, bbox_inches="tight")
plt.show()


## Privacy-safe external scenario figure

The corrected missing-sex fold-median result is the primary model-defined scenario. Encoded sex=0
is a sensitivity scenario. The figure must state that external sex was unavailable and that neither
value represents observed-covariate performance or robust transportability.


In [ ]:
external_path = REPO_ROOT / "results" / "aggregate" / "external_stress_test.csv"
if not external_path.is_file():
    raise FileNotFoundError("Released aggregate external stress-test table is unavailable.")
external = pd.read_csv(external_path)
external
